In [ ]:
import polars as pl
import pandas as pd
import seaborn as sb
import numpy as np
from utils_teste import baixar_extrair_mesclar
from pathlib import Path

In [ ]:
igr_url = 'https://dadosabertos.ans.gov.br/FTP/PDA/IGR/IGR_versao_2023/IGR.csv'
baixar_extrair_mesclar(igr_url, retornar_lazyframe_unificado=True, where="REGISTRO_ANS = 312924")

In [ ]:
lf_origem_igr = pl.scan_parquet(r"c:\GIT\DadosANS\Cargas\processamentos\igr14_20260510_194952\ingestao\downloads\IGR.parquet")

lf_origem_igr.show()

In [ ]:
lf_origem_igr.show(136)

In [ ]:
rpc_url = 'https://dadosabertos.ans.gov.br/FTP/PDA/RPC/'
baixar_extrair_mesclar(rpc_url, retornar_lazyframe_unificado=True, where="CD_OPERADORA = 312924")

In [ ]:
caminho = 'C:\\GIT\\DadosANS\\Cargas\\processamentos\\RPC_lazy_20260510_184902\\ingestao\\_parquet_temporario\\pda-043-rpc-2005_2009.parquet'

pl.read_parquet(caminho).show()

In [ ]:
lf_origem_pda_043_rpc_2005_2009 = pl.scan_parquet(r"c:\GIT\DadosANS\Cargas\processamentos\rpc2_20260510_200818\ingestao\downloads\pda-043-rpc-2005_2009.parquet")

lf_origem_pda_043_rpc_2005_2009.show()

In [ ]:
lf_origem_pda_043_rpc_202602 = pl.scan_parquet(r"c:\GIT\DadosANS\Cargas\processamentos\rpc2_20260510_200818\ingestao\downloads\pda-043-rpc-202602.parquet")
lf_origem_pda_043_rpc_202602.show()

In [ ]:
pasta = Path(r"C:\GIT\DadosANS\Cargas\mescla_ignore\RPC")

lfs = [
    pl.scan_parquet(caminho)
    for caminho in pasta.iterdir()
    if caminho.is_file() and caminho.suffix.lower() == ".parquet"
]

lf_unificado = pl.concat(lfs, how="diagonal_relaxed")

In [ ]:
lf_unificado.sink_parquet(rf'{pasta}\RPC_unificado.parquet')


In [ ]:
pen_url = 'https://dadosabertos.ans.gov.br/FTP/PDA/penalidades_aplicadas_a_operadoras/penalidades_aplicadas_a_operadoras.csv'
baixar_extrair_mesclar(pen_url, retornar_lazyframe_unificado=True)

In [42]:
lf_origem_penalidades_aplicadas_a_operadoras = pl.scan_parquet(r"c:\GIT\DadosANS\Cargas\processamentos\penalidades_20260511_213501\ingestao\downloads\penalidades_aplicadas_a_operadoras.parquet")
lf_origem_penalidades_aplicadas_a_operadoras

In [43]:
colunas_desejaveis = [
    'NR_DEMANDA',
    'NR_PROCESSO',
    'TIPO_PROCESSO',
    'REGISTRO_OPERADORA',
    'STATUS_DEMANDA',
    'DT_PUBLICACAO_1A_FINAL',
    'VL_MULTA_FIXA_1A',
    'VL_TOTAL_APLICADO_1A',
    'DT_CIENCIA_RECONSIDERA_TOTAL',
    'TIPO_DECISAO_2A',
    'VL_MULTA_FIXA_2A',
    'VL_TOTAL_APLICADO_2A',
    'VL_MULTA_FINAL_APLICADA',
    'VL_TOTAL_DESCONTOS',
    'DT_ARQUIVAMENTO',
    'MOTIVO_ARQUIVAMENTO',
    'TIPO_PENALIDADE',
    'DT_SUSPENSAO_ADM',
    'DT_SUSPENSAO_JUD',
    'DT_VENC_GRU',
    'VL_GRU',
    'DE_SITUACAO_GRU',
    'DT_PAGTO_A_VISTA_ANS',
    'VL_PAGO_A_VISTA_ANS',
    'DT_INSCRICAO',
    'INSCRITO_DA',
    'ORIGEM_PAGAMENTO',
    'NR_COMPETENCIA_CARGA',
]

penalidades_sc = (
    lf_origem_penalidades_aplicadas_a_operadoras
    .filter(pl.col("REGISTRO_OPERADORA") == 312924)
    .select(colunas_desejaveis)
    .collect()
)
penalidades_sc = penalidades_sc.with_columns(pl.col(['DT_PUBLICACAO_1A_FINAL',
                                                    'DT_CIENCIA_RECONSIDERA_TOTAL',
                                                    'DT_SUSPENSAO_ADM',
                                                    'DT_SUSPENSAO_JUD',
                                                    'DT_VENC_GRU',
                                                    'DT_PAGTO_A_VISTA_ANS',
                                                    'DT_INSCRICAO',
                                                    'DT_ARQUIVAMENTO']).str.strptime(pl.Date, "%d/%m/%Y", exact=False, strict=False))


penalidades_sc.show()


NR_DEMANDA,NR_PROCESSO,TIPO_PROCESSO,REGISTRO_OPERADORA,STATUS_DEMANDA,DT_PUBLICACAO_1A_FINAL,VL_MULTA_FIXA_1A,VL_TOTAL_APLICADO_1A,DT_CIENCIA_RECONSIDERA_TOTAL,TIPO_DECISAO_2A,VL_MULTA_FIXA_2A,VL_TOTAL_APLICADO_2A,VL_MULTA_FINAL_APLICADA,VL_TOTAL_DESCONTOS,DT_ARQUIVAMENTO,MOTIVO_ARQUIVAMENTO,TIPO_PENALIDADE,DT_SUSPENSAO_ADM,DT_SUSPENSAO_JUD,DT_VENC_GRU,VL_GRU,DE_SITUACAO_GRU,DT_PAGTO_A_VISTA_ANS,VL_PAGO_A_VISTA_ANS,DT_INSCRICAO,INSCRITO_DA,ORIGEM_PAGAMENTO,NR_COMPETENCIA_CARGA
i64,str,str,i64,str,date,i64,i64,date,str,i64,i64,i64,i64,date,str,str,date,date,date,i64,str,date,i64,date,str,str,i64
5379468,"""33910.004280/2022-35""","""Consumidor""",312924,"""Arquivado""",2023-06-25,88000,88000,null,null,0,0,88000,17600,2024-02-27,"""Decisão condenatória com pagam…","""Multa Pecuniária""",null,null,2023-08-16,70400,"""Pago""",2023-08-16,70400,null,"""NAO""","""ANS""",202503
3419328,"""33903.003604/2017-40""","""Consumidor""",312924,"""Arquivado""",2017-05-23,66000,66000,null,"""Provimento Negado""",0,0,0,null,2020-09-24,"""Extinção do Processo (Artigo 5…","""Multa Pecuniária""",null,2019-03-15,2019-02-28,73240,"""Cancelada""",null,null,null,"""NAO""","""ANS""",202503
1664105,"""25785.014384/2012-31""","""Consumidor""",312924,"""Em Cobrança""",2015-08-11,72650,72650,null,"""Provimento Negado""",72650,72650,72650,null,null,null,"""Multa Pecuniária""",null,2017-05-17,2017-01-31,99916,"""Suspenso""",null,null,2017-04-03,"""SIM""","""DA""",202503
2580639,"""25782.009138/2015-30""","""Consumidor""",312924,"""Arquivado""",2016-11-03,66000,66000,null,"""Provimento Negado""",66000,66000,66000,null,2022-10-27,"""Decisão condenatória com pagam…","""Multa Pecuniária""",null,null,2019-02-28,90625,"""Pago""",2020-10-06,99239,null,"""NAO""","""ANS""",202503
4431631,"""33910.026096/2019-41""","""Consumidor""",312924,"""Arquivado""",2022-04-06,79200,79200,null,null,0,0,79200,15840,2024-02-20,"""Decisão condenatória com pagam…","""Multa Pecuniária""",null,null,2023-02-10,63360,"""Pago""",2023-02-10,63360,null,"""NAO""","""ANS""",202503


In [46]:
consulta_valor_ano = penalidades_sc. sql("""
SELECT
    *,
    YEAR(DT_PAGTO_A_VISTA_ANS) AS ANO_PAGTO,
    SUM(valor_pagamento) OVER (
        PARTITION BY YEAR(DT_PAGTO_A_VISTA_ANS)
    ) AS TOTAL_ANO_PAGTO
FROM self
GROUP BY ANO_PAGTO
ORDER BY ANO_PAGTO
""")
consulta_valor_ano

SQLInterfaceError: unsupported function 'year'

In [47]:
consulta_valor_ano = penalidades_sc.group_by(pl.col("DT_PAGTO_A_VISTA_ANS")
                                             .dt
                                             .year().
                                             alias("ANO_PAGTO")).agg(pl.col("VL_PAGO_A_VISTA_ANS")
                                                                     .sum()
                                                                     .alias("TOTAL_ANO_PAGTO")).sort("ANO_PAGTO")
consulta_valor_ano

ANO_PAGTO,TOTAL_ANO_PAGTO
i32,i64
null,0
2013,126041
2014,170918
2015,185878
2016,711519
…,…
2021,2372564
2022,1756637
2023,1848271
